Implementar re-ranking en el retrieval.
-   Probar re-rank en RAG
-   Probar re-rank en grafo
-   Probar re-rank Combinado

INICIALIZACION

In [12]:
import sys
from sentence_transformers import SentenceTransformer

In [2]:
sys.path.append("../src")

In [6]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j
from graph_retrieval.funciones_graph_retrieval import (
    extraer_top_k_entities,
    formatear_tripletas,
)
from conexion_qdrant.conexion_qdrant import ConexionQdrant
from RAG_retrieval.funciones_RAG_retrieval import(
    extraer_info_nodes,
    extraer_info_points,
    filtrar_nodos_por_score,
    textos_para_prompt
)
from funciones_generales import build_prompt
from LLM_interaction import LLM_interaction_functions as llm_funcs
from metricas.metricas_2Wiki import (
    f1_score,
    exact_match_score,
    respuesta_en_nodos_encontrados,
    suporting_facts_en_subgrafo,
    metricas_totales
    )
from output_save.funciones_guardado import guardar_resultados, guardar_registro

# Load Data

In [7]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 100, "train")

In [9]:
ejemplo = dataset_2Wiki[0]

# Qdrant y Neo4j conexion

In [8]:
database_Neo = "2wiki.prueba1"
database_Neo4j = ConexionNeo4j(database_Neo)
qd_client = ConexionQdrant()

In [13]:
collection = "2wikimultihop_prueba1"
embed_model_st = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3080.65it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieval Qdrant

In [17]:
ejemplo

{'_id': '13f5ad2c088c11ebbd6fac1f6bf848b6',
 'type': 'bridge_comparison',
 'question': 'Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?',
 'context': '[["Stuart Rosenberg", ["Stuart Rosenberg (August 11, 1927 \\u2013 March 15, 2007) was an American film and television director whose motion pictures include \\"Cool Hand Luke\\" (1967), \\"Voyage of the Damned\\" (1976), \\"The Amityville Horror\\" (1979), and \\"The Pope of Greenwich Village\\" (1984).", "He was noted for his work with actor Paul Newman."]], ["M\\u00e9diterran\\u00e9e (1963 film)", ["M\\u00e9diterran\\u00e9e is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schl\\u00f6ndorff.", "It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel.", "The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Go

In [11]:
question = ejemplo["question"]
question

'Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?'

Recuperar muchos fragmentos para despues hacer re-ranking

In [15]:
points = qd_client.query_qdrant(collection, embed_model_st, question, 30)
points_clean = extraer_info_points(points)

In [16]:
points_clean


{'45afbf2b-d35f-4b1a-8fd4-646027afbddb': {'text': 'Méditerranée (1963 film). Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.',
  'score': 0.7247652},
 '47d0bdd9-0505-47b9-b3d8-35e782c5fc2b': {'text': 'Move (1970 film). Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss and Geneviève Waïte, and directed by Stuart Rosenberg. The screenplay was written by Joel Lieber and Stanley Hart, adapted from a novel by Lieber.',


## Aplicar re-ranking

In [18]:
from sentence_transformers import CrossEncoder

In [19]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5344.18it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
textos_list = [points_clean[k]["text"] for k,v in points_clean.items()]

In [28]:
textos_list

['Méditerranée (1963 film). Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.',
 'Move (1970 film). Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss and Geneviève Waïte, and directed by Stuart Rosenberg. The screenplay was written by Joel Lieber and Stanley Hart, adapted from a novel by Lieber.',
 'Luciano Salce. Luciano Salce (25 September 1922, in Rome – 17 December 1989, in Rome) was an Italian film director, act

In [30]:
pairs = [[question, text] for text in textos_list ]

In [31]:
scores_rerank = reranker.predict(pairs)

In [32]:
scores_rerank

array([9.9968863e-01, 9.9966109e-01, 1.1233421e-04, 4.7502771e-02,
       1.1745696e-04, 9.2604285e-04, 5.2928299e-05, 4.7442336e-03,
       8.6548930e-04, 2.9320869e-04, 2.6584435e-03, 1.0516833e-02,
       3.1123236e-03, 5.9403568e-03, 4.8236867e-05, 6.9998263e-04,
       5.0084596e-04, 4.7228709e-04, 1.2682032e-04, 2.9968580e-03,
       7.2128227e-04, 4.1724098e-04, 2.2545950e-02, 8.4372783e-05,
       3.2272006e-03, 4.6621257e-04, 3.3690446e-04, 2.0183247e-04,
       5.1132077e-04, 1.1322114e-03], dtype=float32)

Añadir scores del reranking a los resultados del retrieval

In [34]:
points_clean

{'45afbf2b-d35f-4b1a-8fd4-646027afbddb': {'text': 'Méditerranée (1963 film). Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.',
  'score': 0.7247652},
 '47d0bdd9-0505-47b9-b3d8-35e782c5fc2b': {'text': 'Move (1970 film). Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss and Geneviève Waïte, and directed by Stuart Rosenberg. The screenplay was written by Joel Lieber and Stanley Hart, adapted from a novel by Lieber.',


In [37]:
for idx, (k,v) in enumerate(points_clean.items()):
    points_clean[k]["score_rerank"] = scores_rerank[idx]
    # print(k)

In [38]:
points_clean

{'45afbf2b-d35f-4b1a-8fd4-646027afbddb': {'text': 'Méditerranée (1963 film). Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.',
  'score': 0.7247652,
  'score_rerank': np.float32(0.9996886)},
 '47d0bdd9-0505-47b9-b3d8-35e782c5fc2b': {'text': 'Move (1970 film). Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss and Geneviève Waïte, and directed by Stuart Rosenberg. The screenplay was written by Joel Lieber and Stanley